## Imports

In [ ]:
import pandas as pd
import re
import numpy as np
import os
from pathlib import Path
import glob
from scipy import stats
from collections import defaultdict
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import mutual_info_score
from scipy.stats import chi2_contingency, ks_2samp, wasserstein_distance
from functions import *

# WARNINGS
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pkg_resources")

In [2]:
PROJECT_ROOT = find_project_root()
GENERATED_DIR = PROJECT_ROOT / "data" / "generated"

<b style="color:yellow;"> In this cell we load the datasets for each domain. There is approx. 27 datasets per each domain. That is because we have 3 shots (zero, one and few), 3 LLMs and 3 runs for each. </b>

In [3]:
metadata_table = list_generated_datasets(GENERATED_DIR)
metadata_table

Found 81 CSV files.


,domain,model,shot,run,file_path
0,hatecrime,qwen3-coder-30b-a3b-instruct,zero,run3,/Users/veronhoxha/Desktop/master_thesis/data/g...
1,hatecrime,qwen3-coder-30b-a3b-instruct,one,run3,/Users/veronhoxha/Desktop/master_thesis/data/g...
2,hatecrime,qwen3-coder-30b-a3b-instruct,few,run3,/Users/veronhoxha/Desktop/master_thesis/data/g...
3,lending,qwen3-coder-30b-a3b-instruct,zero,run3,/Users/veronhoxha/Desktop/master_thesis/data/g...
4,lending,qwen3-coder-30b-a3b-instruct,one,run3,/Users/veronhoxha/Desktop/master_thesis/data/g...
...,...,...,...,...,...
76,lending,kimi-k2-instruct-0905,one,run2,/Users/veronhoxha/Desktop/master_thesis/data/g...
77,lending,kimi-k2-instruct-0905,few,run2,/Users/veronhoxha/Desktop/master_thesis/data/g...
78,employment,kimi-k2-instruct-0905,zero,run2,/Users/veronhoxha/Desktop/master_thesis/data/g...
79,employment,kimi-k2-instruct-0905,one,run2,/Users/veronhoxha/Desktop/master_thesis/data/g...


In [4]:
kimi_lending_one = metadata_table[
    (metadata_table["domain"] == "lending") &
    (metadata_table["model"] == "kimi-k2-instruct-0905") &
    (metadata_table["shot"] == "one")
]

In [5]:
metadata_table["data"] = None   

for i in range(len(metadata_table)):
    file_path = metadata_table.loc[i, "file_path"]
    df = load_csv_safely(file_path)
    metadata_table.at[i, "data"] = df

metadata_table.iloc[10]

domain                                               hatecrime
model                                    llama-3.1-8b-instruct
shot                                                       one
run                                                       run1
file_path    /Users/veronhoxha/Desktop/master_thesis/data/g...
data                                                      d...
Name: 10, dtype: object

<b style="color:yellow;"> Now it is time to load the real datasets for each domain. </b>

In [6]:
REAL_DATA_DIR = PROJECT_ROOT / "data" / "preprocessed"

employment_real = pd.read_csv(REAL_DATA_DIR / "uk_gender_pay_gap_data_2024_to_2025_preproccesed.csv")
lending_real = pd.read_csv(REAL_DATA_DIR / "year_2024_preprocessed.csv")
hatecrime_real = pd.read_csv(REAL_DATA_DIR / "hate_crime_preprocessed.csv")

print("Load the real employment dataset: ", employment_real.shape)
print("Load the real lending dataset: ", lending_real.shape)
print("Load the real hate crime dataset: ", hatecrime_real.shape)


Load the real employment dataset:  (11239, 18)
Load the real lending dataset:  (12229298, 58)
Load the real hate crime dataset:  (265834, 20)


<b style="color:yellow;"> As it is described in the thesis, sensitive and outcome attributes are defined for each of our domains. </b>

First, we define the sensitive atttributes, then if the sensitive attributes are of type string, the unique values for each sensitive attribute is obtained for the Real and LLM generated datasets, and we map the latter so they match the ones of the real dataset. The mapping are saved as a new column, called {original_column_name}_mapped}

In [7]:
SENSITIVE_ATTRIBUTES = {
    
    # Employment dataset has no sensitive attributes whose values are strings, but I am still providing the code for future reference
    # "employment": ["FemaleBonusPercent", "MaleBonusPercent", "FemaleLowerQuartile", "MaleLowerQuartile", 
    #                "FemaleLowerMiddleQuartile", "MaleLowerMiddleQuartile", "FemaleUpperMiddleQuartile", 
    #                "MaleUpperMiddleQuartile", "FemaleTopQuartile", "MaleTopQuartile"],

    "lending": ["derived_ethnicity", "derived_race", "derived_sex"],

    "hatecrime": ["offender_race", "offender_ethnicity"] 
}


OUTCOME_ATTRIBUTES = {
    # Currently none of the tables have this outcome attribute but it will be created.
    "lending": ["action_taken"],
    "hatecrime": ["offense_name"]
}

In [8]:
# Get the unique values for each sensitive attribute, for the real and LLM generated dataset
# Combine all LLM-generated datasets by domain
llm_data_by_domain = {
    domain: pd.concat(
        [row["data"] for _, row in metadata_table[metadata_table["domain"] == domain].iterrows()],
        ignore_index=True
    )
    for domain in ["lending", "hatecrime"] # Employment dataset has no sensitive attributes whose values are strings
}

real_data_by_domain = {
   # "employment": employment_real, Commented out because the employment dataset has no sensitive attributes whose values are strings
    "lending": lending_real,
    "hatecrime": hatecrime_real
}

print_unique_sensitive_values(SENSITIVE_ATTRIBUTES, real_data_by_domain, llm_data_by_domain)

DOMAIN: LENDING

Sensitive Attribute: derived_ethnicity
  • Real unique values (5):
      ['Ethnicity Not Available', 'Free Form Text Only', 'Hispanic or Latino', 'Joint', 'Not Hispanic or Latino']
  • LLM unique values (119):
      [' "Hispanic or Latino"', ' "Not Hispanic or Latino"', '1', '1-4 Family Home', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '5', '6', '7', '8', '9', 'African American', 'Alaskan Native', 'American Indian', 'American Indian or Alaska Native', 'Application denied', 'Asian', 'Asian or Pacific Islander', 'Black', 'Black or African American', 'Black or African Female', 'Black/African American', 'Brown', 'Central American', 'Ethnicity Not Available', 'European American/White', 'Female', 'File closed for incompleteness', 'HISPIDIAN OR LATINO', 'Hawaiian/Pacific Islander', 'His', 'Hisp or Latino', 'Hisp. or Lati

In [9]:
EMPLOYMENT_PAIRS = [("MaleBonusPercent", "FemaleBonusPercent"), 
                    ("MaleLowerQuartile", "FemaleLowerQuartile"), 
                    ("MaleLowerMiddleQuartile", "FemaleLowerMiddleQuartile"), 
                    ("MaleUpperMiddleQuartile", "FemaleUpperMiddleQuartile"), 
                    ("MaleTopQuartile", "FemaleTopQuartile")]

female_greater_counts = []
for male_col, female_col in EMPLOYMENT_PAIRS:
    df_subset = employment_real[[male_col, female_col]].dropna()
    df_subset[male_col] = pd.to_numeric(df_subset[male_col], errors="coerce")
    df_subset[female_col] = pd.to_numeric(df_subset[female_col], errors="coerce")
    df_subset = df_subset.dropna()
    
    count = (df_subset[female_col] > df_subset[male_col]).sum()
    total = len(df_subset)
    percentage = (count / total * 100) if total > 0 else 0
    female_greater_counts.append({
        "Pair": f"{male_col} vs {female_col}",
        "Count": count,
        "Total": total,
        "Percentage": percentage
    })
    print(f"{male_col} vs {female_col}: {count}/{total} ({percentage:.2f}%)")

avg_percentage = np.mean([d["Percentage"] for d in female_greater_counts])
print(f"\nAverage percentage: {avg_percentage:.2f}%")

print("\n" + "="*80)
print("LLM Generated Employment Datasets")
print("="*80)

llm_results = []
for idx, row in metadata_table.iterrows():
    if row["domain"] != "employment":
        continue
    
    df = row["data"]
    model = row["model"]
    run = row["run"]
    shot = row["shot"]
    
    dataset_percentages = []
    for male_col, female_col in EMPLOYMENT_PAIRS:
        df_subset = df[[male_col, female_col]].dropna()
        df_subset[male_col] = pd.to_numeric(df_subset[male_col], errors="coerce")
        df_subset[female_col] = pd.to_numeric(df_subset[female_col], errors="coerce")
        df_subset = df_subset.dropna()
        
        if len(df_subset) == 0:
            continue
            
        count = (df_subset[female_col] > df_subset[male_col]).sum()
        total = len(df_subset)
        percentage = (count / total * 100) if total > 0 else 0
        dataset_percentages.append(percentage)
    
    if dataset_percentages:
        avg_pct = np.mean(dataset_percentages)
        llm_results.append({
            "Model": model,
            "Run": run,
            "Shot": shot,
            "Avg Percentage": avg_pct
        })

llm_df = pd.DataFrame(llm_results)
if not llm_df.empty:
    print("\nAverage percentage (female > male) per dataset:")
    print(llm_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    print(f"\nOverall average across all LLM datasets: {llm_df['Avg Percentage'].mean():.2f}%")

MaleBonusPercent vs FemaleBonusPercent: 3756/11239 (33.42%)
MaleLowerQuartile vs FemaleLowerQuartile: 6560/11239 (58.37%)
MaleLowerMiddleQuartile vs FemaleLowerMiddleQuartile: 5908/11239 (52.57%)
MaleUpperMiddleQuartile vs FemaleUpperMiddleQuartile: 5200/11239 (46.27%)
MaleTopQuartile vs FemaleTopQuartile: 4157/11239 (36.99%)

Average percentage: 45.52%

LLM Generated Employment Datasets

Average percentage (female > male) per dataset:
                       Model  Run Shot  Avg Percentage
qwen3-coder-30b-a3b-instruct run3 zero           44.54
qwen3-coder-30b-a3b-instruct run3  one           83.57
qwen3-coder-30b-a3b-instruct run3  few           78.90
       llama-3.1-8b-instruct run1 zero            9.08
       llama-3.1-8b-instruct run1  one           72.95
       llama-3.1-8b-instruct run1  few           81.22
qwen3-coder-30b-a3b-instruct run2 zero           46.23
qwen3-coder-30b-a3b-instruct run2  one           88.95
qwen3-coder-30b-a3b-instruct run2  few           81.08
       kim

<b style="color:yellow;"> Mapping the values of the LLM (where it is possible), to the values of the Real Dataset. If it is not possible I am marking it as NaN in the new column. </b>

In [10]:
# Dictionary used for the mapping of the LLM generated datasets to the real datasets, it can be 
# found in the functions.py file. I have manually mapped all the values that were printed above.
# from functions import ATTRIBUTE_ALIASES

In [11]:
# Function that applied the mappings to the LLM generated dataset, so that it can be compared to the real dataset.
apply_mappings_to_all_datasets(metadata_table, ATTRIBUTE_ALIASES)

# Checking if the mappings was succesfull


# test_df = metadata_table.iloc[33].data

# print(test_df["offender_race_Mapped"].dropna().unique())
# print(test_df["offender_ethnicity_Mapped"].dropna().unique())

# print(test_df["derived_ethnicity_Mapped"].dropna().unique())
# print(test_df["derived_race_Mapped"].dropna().unique())
# print(test_df["derived_sex_Mapped"].dropna().unique()


In [12]:
# from functions import find_bad_mappings_for_config


# This function is used to find the 'bad' mappings, that means mappings with a high rate of missing values.
# The threshold is set to 5% of the total number of mappings, in this case. But it can be increased or decreased according 
# to need.

# for i in ["hatecrime", "lending", "employment"]:
#     for j in ["kimi-k2-instruct-0905", "llama-3.1-8b-instruct", "qwen3-coder-30b-a3b-instruct"]:
#         for k in ["run3", "run2", "run1"]:
#             for l in ["zero", "one", "few"]:
#                 bad_mappings_zero = find_bad_mappings_for_config(
#                     metadata_table,
#                     SENSITIVE_ATTRIBUTES,
#                     #domain="hatecrime",
#                     domain=i,
#                     model=j,
#                     #model="llama-3.1-8b-instruct",
#                     #model="qwen3-coder-30b-a3b-instruct",
#                     run=k,
#                     shot=l,
#                     threshold=0.5, 
# )

# Conclusion:
# For kimi-k2-instruct-0905, lending domain, run 1, 2 and 3 for one shot have a high rate of missing values. 
# For kimi-k2-instruct-0905, lending domain, run 1 for zero shot has a high rate of missing values.
# For qwen3-coder-30b-a3b-instruct, lending domain, run 3 and zero shot has a high rate of missing values.

<b style="color:yellow;"> Similar to the sensitive attributes, we are defining the outcome attributes and crreating a new column, in order to have a binary output. </b>

In [13]:
# In this same line of work I print the unique values for real and LLM generated datasets for the outcome attributes
print_unique_outcome_values(
    OUTCOME_ATTRIBUTES,
    real_data_by_domain,
    llm_data_by_domain
)

DOMAIN: LENDING

Outcome Attribute: action_taken
  • Real unique values (11):
      ['File closed for incompleteness', 'Loan originated', 'Application withdrawn', 'Application denied', 'Application approved but not accepted', '6', '7', '8', 8, 7, 6]
  • LLM unique values (429):
      ['1', '2', 'Home Purchase', 'Home Improvement', 'Refinance', 'Home improvement', 'Purchase', 'Refinance - cash out', 'Refinance - no cash out', 'Home purchase', 'Construction', 'File closed for incompleteness', 'Loan originated', 'Application approved but not accepted', 'Loan denied by financial institution', 'Preapproval request granted', 'Loan approved but not accepted', 'Applicant withdrew application', 'Loan approved after submission', 'Denial', 'Loan closed', 'Approval for loan', 'Approvals', 'Application denied by financial institution', 'Application withdrawn by applicant', 'Application withheld by bank', 'Application withdrew by applicant', 'Denial of loan application', 'Loans originated with a non

In [14]:
apply_outcome_mappings_all(real_data_by_domain, metadata_table, OUTCOME_ALIASES)

Outcome columns created on all REAL and LLM datasets successfully.


In [15]:
subset = metadata_table[
    (metadata_table["domain"] == "lending")
].iloc[0]
df_llm = subset["data"]
df_llm["loan_approved"].head()

0    1.0
1    0.0
2    1.0
3    1.0
4    1.0
Name: loan_approved, dtype: float64

In [16]:
# testing
subset = metadata_table[
    (metadata_table["domain"] == "hatecrime")
].iloc[0]
df_hc = subset["data"]
df_hc["is_violent"].head()

0        violent
1        violent
2    non-violent
3    non-violent
4        violent
Name: is_violent, dtype: object

In [17]:
# testing
subset = metadata_table[
    (metadata_table["domain"] == "lending") &
    (metadata_table["model"] == "kimi-k2-instruct-0905") &
    (metadata_table["run"]   == "run1") &
    (metadata_table["shot"]  == "few")
]

df = subset.iloc[0]["data"]

df["loan_approved"].value_counts(dropna=False)

loan_approved
1.0    728
0.0    102
NaN      4
Name: count, dtype: int64

<b style="color:yellow;"> After defining the senstive and outcome attributes, now we calulcate the Metrics. Starting with the domain of <b style="color:red;"> EMPLOYMENT</b>

<b style="color:yellow;"> 1. Base Rate Parity </b>

In [18]:
# Function to calculate the base rate parity, this is for the Employment dataset, in this occasion  only
# the (Male/Female) pairs are needed. 
EMPLOYMENT_PAIRS = [("MaleBonusPercent", "FemaleBonusPercent"), 
                    ("MaleLowerQuartile", "FemaleLowerQuartile"), 
                    ("MaleLowerMiddleQuartile", "FemaleLowerMiddleQuartile"), 
                    ("MaleUpperMiddleQuartile", "FemaleUpperMiddleQuartile"), 
                    ("MaleTopQuartile", "FemaleTopQuartile")]


# TEST
# print(" The average base rate parity for the employment dataset is:", base_rate_parity_employment(employment_real, EMPLOYMENT_PAIRS)[0])
 
# Looping through all the LLM generated datasets and calculating the base rate parity for the Employment dataset

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "employment":
        continue
    
    df = row["data"]        # the actual dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    bp_avg, bp_values = base_rate_parity_employment(df, EMPLOYMENT_PAIRS)

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "bp_avg": bp_avg,
       #"bp_per_pair": bp_values,
    })


employment_bp_results = pd.DataFrame(results)
employment_bp_results.to_csv("employment_e01_base_rate_parity.csv", index=False)
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_e01_base_rate_parity.csv"
employment_bp_results.to_csv(output_path, index=False)


<b style="color:yellow;"> 2. Disparate Impact </b>

In [19]:
results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "employment":
        continue

    df    = row["data"]
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    di_avg, di_values = disparate_impact_employment(df, EMPLOYMENT_PAIRS)

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "di_avg": di_avg,
        # "di_per_pair": di_values,   # optional
    })

employment_di_results = pd.DataFrame(results)

# Testing for real data
# print(disparate_impact_employment(employment_real, EMPLOYMENT_PAIRS)[0])

# Save the results in a csv file
employment_di_results.to_csv("employment_e02_disparate_impact.csv", index=False)
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_e02_disparate_impact.csv"
employment_di_results.to_csv(output_path, index=False)


<b style="color:yellow;"> 3. Mean Difference </b>

In [20]:
results_md = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "employment":
        continue

    df    = row["data"]
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    md_avg, md_values = mean_difference_employment(df, EMPLOYMENT_PAIRS)

    results_md.append({
        "model": model,
        "run": run,
        "shot": shot,
        "md_avg": md_avg,
        # "md_per_pair": md_values,   # keep commented if you only want one column
    })

employment_md_results = pd.DataFrame(results_md)
#print(employment_md_results)

# Real data testing
#print(mean_difference_employment(employment_real, EMPLOYMENT_PAIRS)[0])

# Save the results in a csv file
employment_md_results.to_csv("employment_e03_mean_difference.csv", index=False)
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_e03_mean_difference.csv"
employment_md_results.to_csv(output_path, index=False)

In [21]:
real_metrics_df = pd.DataFrame([{
    "bp_avg": base_rate_parity_employment(employment_real, EMPLOYMENT_PAIRS)[0],
    "di_avg": disparate_impact_employment(employment_real, EMPLOYMENT_PAIRS)[0],
    "md_avg": mean_difference_employment(employment_real, EMPLOYMENT_PAIRS)[0]
}])

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_real_metrics.csv"
real_metrics_df.to_csv(output_path, index=False)

<b style="color:yellow;"> Calculating the metrics for <b style="color:red;"> HATECRIME</b>

<b style="color:yellow;"> 1. Base Rate Parity</b>

In [22]:
results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "hatecrime":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_bp, per_attr_bp, per_attr_probs = base_rate_parity(
        df,
        sensitive_attrs=["offender_ethnicity_Mapped", "offender_race_Mapped"],
        outcome_col="is_violent",
        positive_label="non-violent"
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "brp_overall": overall_bp,
        "brp_ethnicity": per_attr_bp.get("offender_ethnicity_Mapped"),
        "brp_race": per_attr_bp.get("offender_race_Mapped"),
    })

hatecrime_brp_results = pd.DataFrame(results)

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_e01_base_rate_parity.csv"
hatecrime_brp_results.to_csv(output_path, index=False)


<b style="color:yellow;"> 2. Disparate Impact </b>

In [23]:
results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "hatecrime":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_di, per_attr_di, per_attr_di_probs = disparate_impact_multiclass(
        df,
        sensitive_attrs=["offender_ethnicity_Mapped", "offender_race_Mapped"],
        outcome_col="is_violent",
        positive_label="non-violent"
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "di_overall": overall_di,
        "di_race": per_attr_di.get("offender_race_Mapped"),
        "di_ethnicity": per_attr_di.get("offender_ethnicity_Mapped"),
    })

hatecrime_di_results = pd.DataFrame(results)

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_e02_disparate_impact.csv"
hatecrime_di_results.to_csv(output_path, index=False)

<b style="color:yellow;"> 3. BASE RATE </b>

In [24]:
# br_overall_real, br_per_attr_real, br_rates_real = base_rate_multiclass(
#     hatecrime_real,
#     sensitive_attrs=HATECRIME_SENSITIVE,
#     outcome_col=HATECRIME_OUTCOME_COL,
#     positive_label=HATECRIME_POSITIVE_LABEL,
# )

# br_overall_real

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "hatecrime":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_BR, per_attr_BR, per_attr_BR_probs = base_rate_multiclass(
        df,
        sensitive_attrs=["offender_ethnicity_Mapped", "offender_race_Mapped"],
        outcome_col="is_violent",
        positive_label="non-violent"
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "br_overall": overall_BR,
        "br_race": per_attr_BR.get("offender_race_Mapped"),
        "br_ethnicity": per_attr_BR.get("offender_ethnicity_Mapped"),
    })

hatecrime_BR_results = pd.DataFrame(results)
#print(hatecrime_BR_results)

#Save the results in a csv file
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_e03_base_rate.csv"
hatecrime_BR_results.to_csv(output_path, index=False)

In [25]:
HATECRIME_SENSITIVE = ["offender_ethnicity", "offender_race"] # for llm generated datasets the SA have _Mapped suffix
HATECRIME_OUTCOME_COL = "is_violent"          # contains "violent" / "non-violent"
HATECRIME_POSITIVE_LABEL = "non-violent"      # Y = 1

# REAL DATASET BARE RATE PARITY
overall_brp, per_attr_brp, per_attr_brp_probs = base_rate_parity(
    hatecrime_real, 
    HATECRIME_SENSITIVE, 
    HATECRIME_OUTCOME_COL, 
    HATECRIME_POSITIVE_LABEL
)

#REAL DATASET DISPARATE IMPACT
overall_di, per_attr_di, per_attr_di_probs = disparate_impact_multiclass(
    hatecrime_real,
    HATECRIME_SENSITIVE,
    HATECRIME_OUTCOME_COL,
    HATECRIME_POSITIVE_LABEL
)

#REAL DATASET BASE RATE
overall_br, per_attr_br, per_attr_br_probs = base_rate_multiclass(
    hatecrime_real,
    HATECRIME_SENSITIVE,
    HATECRIME_OUTCOME_COL,
    HATECRIME_POSITIVE_LABEL
)


# Put into a DataFrame
real_metrics_hatecrime = pd.DataFrame([{
    "overall_brp": overall_brp,
    "overall_di": overall_di,
    "overall_br": overall_br
}])

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_real_metrics.csv"
real_metrics_hatecrime.to_csv(output_path, index=False)

<b style="color:yellow;"> Calculating the metrics for <b style="color:red;"> LENDING</b>

<b style="color:yellow;"> 1. Base Rate Parity </b>

In [26]:
LENDING_SENSITIVE = ["derived_race", "derived_sex", "derived_ethnicity"]
LENDING_OUTCOME_COL = "loan_approved"          # contains "1" / "0"
LENDING_POSITIVE_LABEL = 1.0      # Y = 1

# Real dataset
# overall_bp_LENDING, per_attr_bp_LENDING, per_attr_probs_LENDING = base_rate_parity(
#     lending_real, 
#     LENDING_SENSITIVE, 
#     LENDING_OUTCOME_COL, 
#     LENDING_POSITIVE_LABEL
# )

# print("Overall BP:", overall_bp_LENDING)
#print("Per-attribute BP:", per_attr_bp)

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "lending":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_bp, per_attr_bp, per_attr_probs = base_rate_parity(
        df,
        sensitive_attrs=["derived_race_Mapped", "derived_sex_Mapped", "derived_ethnicity_Mapped"],
        outcome_col="loan_approved",
        positive_label=1.0
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "bp_overall": overall_bp,
        "bp_race": per_attr_bp.get("derived_race_Mapped"),
        "bp_ethnicity": per_attr_bp.get("derived_ethnicity_Mapped"),
        "bp_sex": per_attr_bp.get("derived_sex_Mapped"),
    })

hatecrime_BRP_results = pd.DataFrame(results)

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_e01_base_rate_parity.csv"
hatecrime_BRP_results.to_csv(output_path, index=False)

<b style="color:yellow;"> 2. Disparate Impact</b>

In [27]:
results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "lending":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_di, per_attr_di, per_attr_di_probs = disparate_impact_multiclass(
        df,
        sensitive_attrs=["derived_race_Mapped", "derived_sex_Mapped", "derived_ethnicity_Mapped"],
        outcome_col="loan_approved",
        positive_label=1.0
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "di_overall": overall_di,
        "di_race": per_attr_di.get("derived_race_Mapped"),
        "di_ethnicity": per_attr_di.get("derived_ethnicity_Mapped"),
        "di_sex": per_attr_di.get("derived_sex_Mapped"),
    })

lending_di_results = pd.DataFrame(results)

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_e02_disparate_impact.csv"
lending_di_results.to_csv(output_path, index=False)

<b style="color:yellow;"> 3. BASE RATE </b>

In [28]:
results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "lending":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_di, per_attr_di, per_attr_di_probs = base_rate_multiclass(
        df,
        sensitive_attrs=["derived_race", "derived_sex", "derived_ethnicity"],
        outcome_col="loan_approved",
        positive_label=1.0
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "br_overall": overall_di,
        "br_race": per_attr_di.get("derived_race"),
        "br_ethnicity": per_attr_di.get("derived_ethnicity"),
        "br_sex": per_attr_di.get("derived_sex"),
    })

lending_br_results = pd.DataFrame(results)

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_e03_base_rate.csv"
lending_br_results.to_csv(output_path, index=False)

In [29]:
# Saving the metrics for the real dataset

LENDING_SENSITIVE = ["derived_race", "derived_sex", "derived_ethnicity"]
LENDING_OUTCOME_COL = "loan_approved"          # contains "1" / "0"
LENDING_POSITIVE_LABEL = 1.0      # Y = 1

# REAL DATASET BARE RATE PARITY
overall_brp_LENDING, per_attr_brp_LENDING, per_attr_brp_probs_LENDING = base_rate_parity(
    lending_real, 
    LENDING_SENSITIVE, 
    LENDING_OUTCOME_COL, 
    LENDING_POSITIVE_LABEL
)

#REAL DATASET DISPARATE IMPACT
overall_di_LENDING, per_attr_di_LENDING, per_attr_di_probs_LENDING = disparate_impact_multiclass(
    lending_real,
    LENDING_SENSITIVE,
    LENDING_OUTCOME_COL,
    LENDING_POSITIVE_LABEL
)

#REAL DATASET BASE RATE
overall_br_LENDING, per_attr_br_LENDING, per_attr_br_probs_LENDING = base_rate_multiclass(
    lending_real,
    LENDING_SENSITIVE,
    LENDING_OUTCOME_COL,
    LENDING_POSITIVE_LABEL
)

real_metrics_lending = pd.DataFrame([{
    "overall_brp": overall_brp_LENDING,
    "overall_di": overall_di_LENDING,
    "overall_br": overall_br_LENDING
}])

output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_real_metrics.csv"
real_metrics_lending.to_csv(output_path, index=False)